# Calendar API Capabilities Playground

This notebook is for understanding what a calendar provider can give Dayli and what Dayli will eventually need to send back.

Use it to think like a product + systems architect:

- What can I read from a calendar API?
- What can I write or update?
- What fields matter for a conversational scheduling assistant?
- What Dayli features depend on which calendar capabilities?

## Dayli objective

Dayli is not just a calendar CRUD app.

It needs calendar APIs to support workflows like:

- create a schedule from natural language
- inspect existing events before suggesting changes
- move or resize existing events
- detect conflicts and free space
- preserve user intent during conversational edits
- later: learn preferences from accepted/rejected plans

In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pprint import pprint


## Core capability model

This is a simple product-oriented model of the kinds of operations Dayli will need from Google Calendar or Outlook.

In [ ]:
@dataclass(slots=True)
class CalendarCapability:
    name: str
    why_dayli_needs_it: str
    example_user_request: str
    provider_examples: list[str] = field(default_factory=list)


capabilities = [
    CalendarCapability(
        name="List calendars",
        why_dayli_needs_it="So the app can know which calendar to read/write, such as Personal vs Work.",
        example_user_request="Put all personal plans on my Personal calendar.",
        provider_examples=["Google: calendarList.list", "Microsoft Graph: /me/calendars"],
    ),
    CalendarCapability(
        name="Read events in a time range",
        why_dayli_needs_it="Needed to see what already exists before proposing a plan or checking free time.",
        example_user_request="I want to work from 9-12, gym at 6, dinner at 8.",
        provider_examples=["Google: events.list", "Microsoft Graph: /calendarView"],
    ),
    CalendarCapability(
        name="Create event",
        why_dayli_needs_it="Needed to turn a proposed Dayli schedule into actual calendar events.",
        example_user_request="Schedule gym at 6.",
        provider_examples=["Google: events.insert", "Microsoft Graph: POST /events"],
    ),
    CalendarCapability(
        name="Update event",
        why_dayli_needs_it="Needed for conversational edits like move, rename, or resize.",
        example_user_request="Move my gym to tomorrow.",
        provider_examples=["Google: events.patch", "Microsoft Graph: PATCH /events/{id}"],
    ),
    CalendarCapability(
        name="Delete event",
        why_dayli_needs_it="Needed if the user says to cancel or remove an event.",
        example_user_request="Delete dinner.",
        provider_examples=["Google: events.delete", "Microsoft Graph: DELETE /events/{id}"],
    ),
    CalendarCapability(
        name="Read attendees and metadata",
        why_dayli_needs_it="Needed to tell whether an event is flexible, shared, important, or hard to move.",
        example_user_request="Make my morning less busy.",
        provider_examples=["attendees", "organizer", "location", "status"],
    ),
]

for capability in capabilities:
    pprint(asdict(capability))
    print()

## What data Dayli wants from an event

A calendar provider returns many fields, but not all of them matter equally.

For Dayli, these are the most important ones.

In [ ]:
important_event_fields = {
    "id": "Needed to update, move, or delete the exact event later.",
    "summary/title": "Human-readable event name like Gym, Deep Work, Dinner.",
    "start": "Used for timeline layout, free/busy checks, and edits.",
    "end": "Needed to compute duration, overlap, and available gaps.",
    "timezone": "Important when users say tomorrow, morning, or travel across zones.",
    "location": "Useful for travel time, commute buffers, and context-aware planning.",
    "description": "Can hold notes, meeting context, or instructions.",
    "attendees": "Helps detect whether an event is shared and harder to move.",
    "organizer": "Useful for determining ownership and edit confidence.",
    "status": "Cancelled or tentative events should be treated differently.",
    "recurrence": "Important for repeating habits and repeating edits.",
    "calendar_id": "Needed when the user has multiple calendars.",
}

important_event_fields

## Sample provider event shapes

These are simplified examples so you can see what real calendar payloads tend to look like.

In [ ]:
google_event_example = {
    "id": "abc123",
    "summary": "Gym",
    "start": {"dateTime": "2026-04-10T18:00:00+01:00", "timeZone": "Europe/Dublin"},
    "end": {"dateTime": "2026-04-10T19:00:00+01:00", "timeZone": "Europe/Dublin"},
    "organizer": {"email": "me@example.com"},
    "attendees": [],
    "status": "confirmed",
    "location": "Local gym",
}

graph_event_example = {
    "id": "evt_456",
    "subject": "Gym",
    "start": {"dateTime": "2026-04-10T18:00:00", "timeZone": "GMT Standard Time"},
    "end": {"dateTime": "2026-04-10T19:00:00", "timeZone": "GMT Standard Time"},
    "organizer": {"emailAddress": {"address": "me@example.com"}},
    "attendees": [],
    "isCancelled": False,
    "location": {"displayName": "Local gym"},
}

print("Google event sample")
print(json.dumps(google_event_example, indent=2))
print()
print("Microsoft Graph event sample")
print(json.dumps(graph_event_example, indent=2))

## Translate provider data into a Dayli-friendly shape

This is the kind of normalized event model your backend should eventually use internally.

In [ ]:
def normalize_google_event(event: dict) -> dict:
    return {
        "provider": "google",
        "provider_event_id": event["id"],
        "title": event.get("summary", "Untitled"),
        "start": event["start"].get("dateTime"),
        "end": event["end"].get("dateTime"),
        "timezone": event["start"].get("timeZone"),
        "location": event.get("location"),
        "attendee_count": len(event.get("attendees", [])),
        "status": event.get("status"),
    }


normalize_google_event(google_event_example)

## Which API capabilities map to which Dayli product features?

This is the most important systems-thinking section of the notebook.

In [ ]:
feature_dependency_map = {
    "Create my day from text": [
        "Read events in a time range",
        "Create event",
    ],
    "Move my gym to tomorrow": [
        "Read events in a time range",
        "Update event",
    ],
    "Make my morning less busy": [
        "Read events in a time range",
        "Read attendees and metadata",
        "Update event",
    ],
    "Put work and personal plans on different calendars": [
        "List calendars",
        "Create event",
    ],
    "Cancel dinner": [
        "Delete event",
    ],
}

feature_dependency_map

## Questions you should ask when integrating a calendar API

These are the implementation questions that matter for Dayli.

In [ ]:
integration_questions = [
    "How do I authenticate the user with OAuth?",
    "How do I know which calendar is the default write target?",
    "How do I fetch events in a date range efficiently?",
    "How do I distinguish flexible personal blocks from shared meetings?",
    "How do I handle recurring events safely?",
    "How do I map provider-specific timezone formats into one internal format?",
    "How do I avoid accidental writes before the user confirms?",
    "How do I store provider event IDs so edits can target the right event later?",
]

for question in integration_questions:
    print(f"- {question}")

## Minimal Dayli calendar integration plan

If you want the smallest useful integration for the product, start here.

In [ ]:
minimal_plan = [
    "1. OAuth login with Google Calendar",
    "2. Read events for a selected day",
    "3. Normalize provider events into one Dayli event model",
    "4. Use that data for conflict detection and free-slot discovery",
    "5. Create preview changes before any write",
    "6. Write confirmed changes with create/update/delete",
    "7. Persist provider event IDs for future conversational edits",
]

for step in minimal_plan:
    print(step)

## What to learn from this notebook

The main lesson is this:

Dayli does not need the calendar API just to store events.
It needs the calendar API as context for reasoning, conflict detection, and safe conversational editing.

That means your integration design should focus on:

- reading calendar state well
- normalizing provider data well
- writing only after deterministic validation
- keeping stable IDs so conversational edits work reliably